# Nutrition5k mass estimation: pseudo-depth v3


In [1]:
from pathlib import Path
import json
import math
import os
import sys
import time
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor, Pool
from ultralytics import YOLO


c:\Projects\FoodProject\FoodProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Project paths and settings

In [2]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if all((path / name).exists() for name in ("DepthModule", "mass_estimation", "nutrition5k", "models")):
            return path
    raise RuntimeError("Project root was not found. Run the notebook from inside the cloned repository.")


def env_path(name: str, default: Path) -> Path:
    return Path(os.getenv(name, default)).expanduser()


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "DepthModule"))
from depth_module import DepthAnything3Module

MASS_DIR = PROJECT_ROOT / "mass_estimation"
RESULTS_DIR = MASS_DIR / "pseudo-depth-v3-try1"
MAPPING_DIR = PROJECT_ROOT / "class_mappings"

NUTRITION_CSV = env_path("NUTRITION_CSV", PROJECT_ROOT / "nutrition5k" / "dish_nutrition_values.csv")
OVERHEAD_DIR = env_path("OVERHEAD_DIR", PROJECT_ROOT / "nutrition5k" / "imagery" / "realsense_overhead")
SEG_MODEL_PATH = env_path("SEG_MODEL_PATH", PROJECT_ROOT / "models" / "yolo_food_seg.pt")

_cls_default = PROJECT_ROOT / "models" / "yolo_cls.pt"
_cls_override = os.getenv("CLS_MODEL_PATH")
CLS_MODEL_PATH = Path(_cls_override).expanduser() if _cls_override else (_cls_default if _cls_default.exists() else None)
DEPTH_MODEL_ID = os.getenv("DEPTH_MODEL", "depth-anything-v3-base")

FEATURES_CSV = RESULTS_DIR / "catboost_pseudodepth_v3_features.csv"
MODEL_PATH = RESULTS_DIR / "catboost_pseudodepth_v3.cbm"
METRICS_PATH = RESULTS_DIR / "catboost_pseudodepth_v3_metrics.json"

RANDOM_STATE = 42
MAX_DISHES = None
RESUME_FEATURE_CACHE = True
YOLO_CONF = 0.25
YOLO_IMGSZ = 640
CACHE_EVERY_N_ROWS = 25

CATBOOST_PARAMS = dict(
    loss_function="MAE",
    eval_metric="MAE",
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=8,
    random_seed=RANDOM_STATE,
    od_type="Iter",
    od_wait=150,
    verbose=100,
)

required_paths = [NUTRITION_CSV, OVERHEAD_DIR, SEG_MODEL_PATH, MAPPING_DIR]
missing = [str(p.relative_to(PROJECT_ROOT) if p.is_relative_to(PROJECT_ROOT) else p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required project paths: " + ", ".join(missing))

print("Project root:", PROJECT_ROOT)
print("Nutrition CSV:", NUTRITION_CSV.relative_to(PROJECT_ROOT))
print("Overhead images:", OVERHEAD_DIR.relative_to(PROJECT_ROOT))
print("Segmentation model:", SEG_MODEL_PATH.relative_to(PROJECT_ROOT))
print("Classification model:", CLS_MODEL_PATH.relative_to(PROJECT_ROOT) if CLS_MODEL_PATH else "disabled")
print("Depth model:", DEPTH_MODEL_ID)
print("Outputs:", RESULTS_DIR.relative_to(PROJECT_ROOT))


Project root: c:\Projects\FoodProject\FoodProject
Nutrition CSV: nutrition5k\dish_nutrition_values.csv
Overhead images: nutrition5k\imagery\realsense_overhead
Segmentation model: models\yolo_food_seg.pt
Classification model: models\yolo_cls.pt
Depth model: depth-anything-v3-base
Outputs: mass_estimation\pseudo-depth-v3-try1


## 3. Metadata


In [3]:
seg_map = pd.read_csv(MAPPING_DIR / "foodseg103_density_groups.csv")
seg_by_id = seg_map.set_index("class_id").to_dict("index")
food101_map = pd.read_csv(MAPPING_DIR / "food101_dish_groups.csv")
food101_group_by_name = dict(zip(food101_map["class_name"].astype(str), food101_map["dish_group"].astype(str)))

density_groups = sorted(seg_map.loc[seg_map["use_for_mask"].astype(str).str.lower() == "true", "density_group"].unique())
labels = (
    pd.read_csv(NUTRITION_CSV)[["dish_id", "mass"]]
    .dropna()
    .assign(rgb_path=lambda df: df["dish_id"].astype(str).map(lambda dish_id: OVERHEAD_DIR / dish_id / "rgb.png"))
)
labels = labels[labels["rgb_path"].map(Path.exists)].copy()
if MAX_DISHES is not None:
    labels = labels.head(MAX_DISHES).copy()

print("Density groups:", density_groups)
print("Food101 groups:", sorted(food101_map["dish_group"].unique()))
print(f"Rows with rgb.png and mass target: {len(labels):,}")
labels.head()


Density groups: ['bread', 'dairy_dessert', 'dairy_fat', 'dessert', 'fruit', 'fruit_dried', 'fruit_fat', 'leafy_veg', 'liquid', 'meat', 'meat_processed', 'mixed_main', 'mushroom', 'nuts', 'protein', 'sauce', 'seafood', 'seaweed', 'starch', 'starch_legume', 'starch_main', 'starch_veg', 'unknown', 'vegetable']
Food101 groups: ['bread', 'breakfast', 'dairy_dessert', 'dairy_fat', 'dessert', 'meat_main', 'mixed_main', 'protein', 'salad', 'sandwich', 'sauce', 'seafood', 'soup_liquid', 'starch', 'starch_legume', 'starch_main']
Rows with rgb.png and mass target: 3,244


,dish_id,mass,rgb_path
0,dish_1561662216,193.0,c:\Projects\FoodProject\FoodProject\nutrition5...
2,dish_1561662054,292.0,c:\Projects\FoodProject\FoodProject\nutrition5...
3,dish_1562008979,290.0,c:\Projects\FoodProject\FoodProject\nutrition5...
4,dish_1560455030,103.0,c:\Projects\FoodProject\FoodProject\nutrition5...
5,dish_1558372433,143.0,c:\Projects\FoodProject\FoodProject\nutrition5...


## 4. Feature engineering helpers


In [4]:
def resize_nearest(mask: np.ndarray, shape_hw: Tuple[int, int]) -> np.ndarray:
    if mask.shape[:2] == shape_hw:
        return mask.astype(bool)
    return cv2.resize(mask.astype(np.uint8), (shape_hw[1], shape_hw[0]), interpolation=cv2.INTER_NEAREST).astype(bool)

def resize_depth(depth: np.ndarray, shape_hw: Tuple[int, int]) -> np.ndarray:
    depth = np.asarray(depth, dtype=np.float32)
    if depth.shape[:2] == shape_hw:
        return depth
    return cv2.resize(depth, (shape_hw[1], shape_hw[0]), interpolation=cv2.INTER_LINEAR)

def normalize_depth(depth: np.ndarray) -> Dict[str, np.ndarray]:
    depth = np.asarray(depth, dtype=np.float32)
    finite = depth[np.isfinite(depth)]
    if finite.size == 0:
        z = np.zeros_like(depth, dtype=np.float32)
        return {"raw": z, "p05p95": z, "iqr_z": z}
    p05, p25, p50, p75, p95 = np.percentile(finite, [5, 25, 50, 75, 95])
    scale = max(float(p95 - p05), 1e-6)
    iqr = max(float(p75 - p25), 1e-6)
    p05p95 = np.clip((depth - p05) / scale, 0.0, 1.0).astype(np.float32)
    iqr_z = np.clip((depth - p50) / iqr, -5.0, 5.0).astype(np.float32)
    return {"raw": depth, "p05p95": p05p95, "iqr_z": iqr_z}

def mask_shape_features(mask: np.ndarray, prefix: str) -> Dict[str, float]:
    mask_u8 = mask.astype(np.uint8)
    area = int(mask_u8.sum())
    out = {f"{prefix}_area_px": area}
    if area == 0:
        out.update({
            f"{prefix}_perimeter": 0.0,
            f"{prefix}_compactness": 0.0,
            f"{prefix}_solidity": 0.0,
            f"{prefix}_equiv_diameter": 0.0,
        })
        return out
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter = float(sum(cv2.arcLength(c, True) for c in contours))
    hull_area = 0.0
    for c in contours:
        if len(c) >= 3:
            hull_area += float(cv2.contourArea(cv2.convexHull(c)))
    out.update({
        f"{prefix}_perimeter": perimeter,
        f"{prefix}_compactness": float(area / max(perimeter * perimeter, 1e-6)),
        f"{prefix}_solidity": float(area / max(hull_area, 1e-6)),
        f"{prefix}_equiv_diameter": float(math.sqrt(4.0 * area / math.pi)),
    })
    return out


In [5]:
def predict_seg_instances(seg_model: YOLO, image: Image.Image) -> Tuple[List[Dict[str, object]], Dict[str, float]]:
    image_np = np.array(image.convert("RGB"))
    h, w = image_np.shape[:2]
    result = seg_model.predict(image_np, retina_masks=True, conf=YOLO_CONF, imgsz=YOLO_IMGSZ, verbose=False)[0]
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return [], {"n_masks_raw": 0, "n_masks_kept": 0, "seg_conf_mean": 0.0, "seg_conf_max": 0.0}

    raw_masks = result.masks.data.cpu().numpy() > 0.5
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy().astype(float)

    instances = []
    for raw_mask, class_id, conf in zip(raw_masks, classes, confs):
        meta = seg_by_id.get(int(class_id), {})
        use_for_mask = str(meta.get("use_for_mask", "true")).lower() == "true"
        if not use_for_mask:
            continue
        mask = resize_nearest(raw_mask, (h, w))
        area = int(mask.sum())
        if area == 0:
            continue
        instances.append({
            "mask": mask,
            "class_id": int(class_id),
            "class_name": str(meta.get("class_name", seg_model.names.get(int(class_id), class_id))),
            "density_group": str(meta.get("density_group", "unknown")),
            "conf": float(conf),
            "area": area,
        })

    stats = {
        "n_masks_raw": int(len(raw_masks)),
        "n_masks_kept": int(len(instances)),
        "seg_conf_mean": float(np.mean(confs)) if len(confs) else 0.0,
        "seg_conf_max": float(np.max(confs)) if len(confs) else 0.0,
    }
    return instances, stats

def predict_food101(cls_model: Optional[YOLO], image: Image.Image, topk: int = 5) -> Dict[str, object]:
    out = {"food101_top1": "unknown", "food101_top1_conf": 0.0, "food101_dish_group": "unknown", "food101_entropy": 0.0}
    for k in range(2, topk + 1):
        out[f"food101_top{k}"] = "unknown"
        out[f"food101_top{k}_conf"] = 0.0
    if cls_model is None:
        return out
    result = cls_model.predict(np.array(image.convert("RGB")), imgsz=YOLO_IMGSZ, verbose=False)[0]
    probs = getattr(result, "probs", None)
    if probs is None:
        return out
    top_ids = list(probs.top5[:topk])
    top_confs = [float(x) for x in probs.top5conf[:topk]]
    for i, (class_id, conf) in enumerate(zip(top_ids, top_confs), start=1):
        name = str(cls_model.names.get(int(class_id), class_id))
        out[f"food101_top{i}"] = name
        out[f"food101_top{i}_conf"] = conf
        if i == 1:
            out["food101_dish_group"] = food101_group_by_name.get(name, "unknown")
    conf_arr = np.asarray(top_confs, dtype=np.float32)
    conf_arr = conf_arr / max(float(conf_arr.sum()), 1e-6)
    out["food101_entropy"] = float(-(conf_arr * np.log(conf_arr + 1e-9)).sum())
    return out


In [6]:
def height_features(mask: np.ndarray, depth_norm: np.ndarray, prefix: str, ring_fracs=(0.015, 0.035, 0.07)) -> Dict[str, float]:
    h, w = mask.shape[:2]
    out = {}
    food_values = depth_norm[mask]
    food_values = food_values[np.isfinite(food_values)]
    if food_values.size == 0:
        for frac in ring_fracs:
            tag = f"{prefix}_ring{int(frac * 1000):03d}"
            out[f"{tag}_plate_depth"] = 0.0
            out[f"{tag}_vol_plate_minus_food"] = 0.0
            out[f"{tag}_vol_food_minus_plate"] = 0.0
            out[f"{tag}_mean_abs_height"] = 0.0
        return out

    for frac in ring_fracs:
        kernel_size = max(5, int(round(min(h, w) * frac)))
        if kernel_size % 2 == 0:
            kernel_size += 1
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        dilated = cv2.dilate(mask.astype(np.uint8), kernel, iterations=1).astype(bool)
        ring = dilated & ~mask
        ring_values = depth_norm[ring]
        ring_values = ring_values[np.isfinite(ring_values)]
        if ring_values.size < 20:
            ring_values = depth_norm[~mask]
            ring_values = ring_values[np.isfinite(ring_values)]
        plate_depth = float(np.median(ring_values)) if ring_values.size else float(np.median(depth_norm[np.isfinite(depth_norm)]))
        plate_minus_food = np.clip(plate_depth - food_values, 0, None)
        food_minus_plate = np.clip(food_values - plate_depth, 0, None)
        if plate_minus_food.size:
            plate_minus_food = np.clip(plate_minus_food, 0, np.percentile(plate_minus_food, 95))
        if food_minus_plate.size:
            food_minus_plate = np.clip(food_minus_plate, 0, np.percentile(food_minus_plate, 95))
        tag = f"{prefix}_ring{int(frac * 1000):03d}"
        out[f"{tag}_plate_depth"] = plate_depth
        out[f"{tag}_vol_plate_minus_food"] = float(plate_minus_food.sum())
        out[f"{tag}_vol_food_minus_plate"] = float(food_minus_plate.sum())
        out[f"{tag}_mean_abs_height"] = float(np.mean(np.abs(food_values - plate_depth)))
        out[f"{tag}_p75_abs_height"] = float(np.percentile(np.abs(food_values - plate_depth), 75))
        out[f"{tag}_p95_abs_height"] = float(np.percentile(np.abs(food_values - plate_depth), 95))
    return out

def compute_image_features(instances: List[Dict[str, object]], depth: np.ndarray, image_size: Tuple[int, int]) -> Dict[str, object]:
    h, w = image_size
    depth = resize_depth(depth, (h, w))
    depth_versions = normalize_depth(depth)

    out: Dict[str, object] = {"image_h": h, "image_w": w, "image_area_px": h * w}
    for group in density_groups:
        out[f"area_group_{group}"] = 0.0
        out[f"count_group_{group}"] = 0.0
        out[f"pvol_group_{group}"] = 0.0

    if not instances:
        out.update({"area_px": 0, "area_ratio": 0.0, "n_unique_seg_classes": 0, "dominant_density_group": "unknown", "dominant_seg_class": "unknown"})
        out.update(mask_shape_features(np.zeros((h, w), dtype=bool), "union"))
        return out

    union_mask = np.logical_or.reduce([inst["mask"] for inst in instances])
    area_px = int(union_mask.sum())
    out.update({
        "area_px": area_px,
        "sqrt_area": float(math.sqrt(area_px)),
        "log_area": float(math.log1p(area_px)),
        "area_ratio": float(area_px / max(h * w, 1)),
        "n_unique_seg_classes": int(len(set(inst["class_id"] for inst in instances))),
    })
    out.update(mask_shape_features(union_mask, "union"))

    instances_sorted = sorted(instances, key=lambda x: int(x["area"]), reverse=True)
    for rank in range(1, 4):
        if len(instances_sorted) >= rank:
            inst = instances_sorted[rank - 1]
            out[f"top{rank}_seg_class"] = inst["class_name"]
            out[f"top{rank}_density_group"] = inst["density_group"]
            out[f"top{rank}_area_ratio"] = float(inst["area"] / max(area_px, 1))
        else:
            out[f"top{rank}_seg_class"] = "unknown"
            out[f"top{rank}_density_group"] = "unknown"
            out[f"top{rank}_area_ratio"] = 0.0
    out["dominant_seg_class"] = out["top1_seg_class"]
    out["dominant_density_group"] = out["top1_density_group"]

    p05p95 = depth_versions["p05p95"]
    out.update(height_features(union_mask, p05p95, "union_p05p95"))
    out.update(height_features(union_mask, depth_versions["iqr_z"], "union_iqrz"))

    for inst in instances:
        group = str(inst["density_group"])
        mask = inst["mask"]
        out[f"area_group_{group}"] = float(out.get(f"area_group_{group}", 0.0) + int(inst["area"]))
        out[f"count_group_{group}"] = float(out.get(f"count_group_{group}", 0.0) + 1.0)
        hf = height_features(mask, p05p95, "tmp")
        out[f"pvol_group_{group}"] = float(out.get(f"pvol_group_{group}", 0.0) + hf.get("tmp_ring035_vol_plate_minus_food", 0.0) + hf.get("tmp_ring035_vol_food_minus_plate", 0.0))

    for group in density_groups:
        out[f"area_ratio_group_{group}"] = float(out.get(f"area_group_{group}", 0.0) / max(area_px, 1))

    return out


## 5. Feature extraction


In [7]:
seg_model = YOLO(str(SEG_MODEL_PATH))
cls_model = YOLO(str(CLS_MODEL_PATH)) if CLS_MODEL_PATH else None
depth_engine = DepthAnything3Module(model_id=DEPTH_MODEL_ID)

feature_rows: List[Dict[str, object]] = []
done_ids = set()
if RESUME_FEATURE_CACHE and FEATURES_CSV.exists():
    cached = pd.read_csv(FEATURES_CSV)
    feature_rows = cached.to_dict("records")
    done_ids = set(cached["dish_id"].astype(str))
    print(f"Loaded cached feature rows: {len(done_ids):,}")

pending = labels[~labels["dish_id"].astype(str).isin(done_ids)].copy()
print(f"Pending dishes: {len(pending):,}")


Pending dishes: 3,244


In [8]:
errors = []
started_at = time.time()

for _, row in tqdm(list(pending.iterrows()), total=len(pending)):
    dish_id = str(row["dish_id"])
    image_path = Path(row["rgb_path"])
    try:
        image = Image.open(image_path).convert("RGB")
        width, height = image.size
        instances, seg_stats = predict_seg_instances(seg_model, image)
        depth = np.asarray(depth_engine.get_depth_matrix(str(image_path)), dtype=np.float32)
        feature_rows.append({
            "dish_id": dish_id,
            "image_path": str(image_path.relative_to(PROJECT_ROOT)),
            "mass": float(row["mass"]),
            **seg_stats,
            **compute_image_features(instances, depth, (height, width)),
            **predict_food101(cls_model, image),
        })
    except Exception as exc:
        errors.append({"dish_id": dish_id, "error": repr(exc)})

    if len(feature_rows) % CACHE_EVERY_N_ROWS == 0:
        pd.DataFrame(feature_rows).to_csv(FEATURES_CSV, index=False)

features = pd.DataFrame(feature_rows)
features.to_csv(FEATURES_CSV, index=False)
print(f"Feature rows: {len(features):,}")
print(f"Errors: {len(errors):,}")
print(f"Elapsed minutes: {(time.time() - started_at) / 60:.1f}")
features.head()


100%|██████████| 3244/3244 [2:09:06<00:00,  2.39s/it]  


Feature rows: 3,244
Errors: 0
Elapsed minutes: 129.1


,dish_id,image_path,mass,n_masks_raw,n_masks_kept,seg_conf_mean,seg_conf_max,image_h,image_w,image_area_px,...,food101_dish_group,food101_entropy,food101_top2,food101_top2_conf,food101_top3,food101_top3_conf,food101_top4,food101_top4_conf,food101_top5,food101_top5_conf
0,dish_1561662216,nutrition5k\imagery\realsense_overhead\dish_15...,193.0,19,19,0.391842,0.900309,480,640,307200,...,starch_main,1.540568,seaweed_salad,0.101885,ceviche,0.075266,guacamole,0.065303,sushi,0.056953
1,dish_1561662054,nutrition5k\imagery\realsense_overhead\dish_15...,292.0,20,20,0.469448,0.986213,480,640,307200,...,salad,1.544753,seaweed_salad,0.094025,risotto,0.085738,greek_salad,0.071019,ceviche,0.065746
2,dish_1562008979,nutrition5k\imagery\realsense_overhead\dish_15...,290.0,15,15,0.430986,0.807133,480,640,307200,...,starch_main,1.241351,seaweed_salad,0.060600,tacos,0.046372,guacamole,0.040838,falafel,0.037647
3,dish_1560455030,nutrition5k\imagery\realsense_overhead\dish_15...,103.0,12,11,0.364814,0.540256,480,640,307200,...,seafood,1.489831,tacos,0.092296,seaweed_salad,0.065655,frozen_yogurt,0.040686,sushi,0.034716
4,dish_1558372433,nutrition5k\imagery\realsense_overhead\dish_15...,143.0,8,8,0.492526,0.707596,480,640,307200,...,dairy_dessert,1.555334,lobster_bisque,0.073040,panna_cotta,0.063794,foie_gras,0.053038,ice_cream,0.049864


## 6. CatBoost training


In [10]:
features = pd.read_csv(FEATURES_CSV)
features = features.replace([np.inf, -np.inf], np.nan).dropna(subset=["mass"])

drop_cols = ["dish_id", "image_path", "mass"]
feature_cols = [c for c in features.columns if c not in drop_cols]
cat_cols = [
    c for c in feature_cols
    if pd.api.types.is_object_dtype(features[c])
    or pd.api.types.is_string_dtype(features[c])
    or isinstance(features[c].dtype, pd.CategoricalDtype)
]
num_cols = [c for c in feature_cols if c not in cat_cols]

features[cat_cols] = features[cat_cols].fillna("unknown").astype(str)
features[num_cols] = features[num_cols].fillna(0.0)

train_df, valid_df = train_test_split(features, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_valid = train_df[feature_cols], valid_df[feature_cols]
y_train = np.log1p(train_df["mass"].astype(float))
y_valid_log = np.log1p(valid_df["mass"].astype(float))
y_valid = valid_df["mass"].astype(float)

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
valid_pool = Pool(X_valid, y_valid_log, cat_features=cat_cols)
print(f"Train rows: {len(train_df):,}; valid rows: {len(valid_df):,}")
print(f"Features: {len(feature_cols)}; categorical: {len(cat_cols)}")


Train rows: 2,595; valid rows: 649
Features: 172; categorical: 14


In [ ]:
model = CatBoostRegressor(**CATBOOST_PARAMS)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

pred = np.clip(np.expm1(model.predict(valid_pool)), 0, None)
valid_report = valid_df[["dish_id", "mass"]].copy()
valid_report["pred"] = pred
valid_report["abs_error"] = np.abs(valid_report["mass"] - valid_report["pred"])
valid_report["mass_bin"] = pd.cut(valid_report["mass"], bins=[0, 100, 250, 500, np.inf], labels=["0-100", "100-250", "250-500", "500+"])

metrics = {
    "valid_mae_g": float(mean_absolute_error(y_valid, pred)),
    "valid_rmse_g": float(math.sqrt(mean_squared_error(y_valid, pred))),
    "valid_r2": float(r2_score(y_valid, pred)),
    "baseline_median_mae_g": float(mean_absolute_error(y_valid, np.full_like(y_valid, train_df["mass"].median(), dtype=float))),
    "bin_mae_g": {str(k): float(v) for k, v in valid_report.groupby("mass_bin", observed=False)["abs_error"].mean().to_dict().items()},
    "train_rows": int(len(train_df)),
    "valid_rows": int(len(valid_df)),
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "model_path": str(MODEL_PATH.relative_to(PROJECT_ROOT)),
    "features_csv": str(FEATURES_CSV.relative_to(PROJECT_ROOT)),
}

model.save_model(MODEL_PATH)
METRICS_PATH.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metrics, indent=2, ensure_ascii=False))


0:	learn: 0.6600752	test: 0.6579122	best: 0.6579122 (0)	total: 206ms	remaining: 10m 16s
100:	learn: 0.3686231	test: 0.4300681	best: 0.4299810 (99)	total: 5.43s	remaining: 2m 36s
200:	learn: 0.3118224	test: 0.4243016	best: 0.4242659 (199)	total: 10.5s	remaining: 2m 26s
300:	learn: 0.2708534	test: 0.4228821	best: 0.4226965 (295)	total: 15.6s	remaining: 2m 19s
400:	learn: 0.2376299	test: 0.4217751	best: 0.4217346 (399)	total: 20.7s	remaining: 2m 13s
500:	learn: 0.2094831	test: 0.4202386	best: 0.4202123 (499)	total: 25.9s	remaining: 2m 9s
600:	learn: 0.1894468	test: 0.4199609	best: 0.4195612 (548)	total: 30.9s	remaining: 2m 3s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.4195612145
bestIteration = 548

Shrink model to first 549 iterations.
{
  "valid_mae_g": 80.35000685705523,
  "valid_rmse_g": 115.81345082518452,
  "valid_r2": 0.45924882351567053,
  "baseline_median_mae_g": 123.34360554699538,
  "bin_mae_g": {
    "0-100": 47.25427990062424,
    "100-250": 56.39046

: 